# Frequency-Coupled Gradient Increments

This notebook implements the complete experimental suite. The essential correction is that **the coordinates are never swapped**, and the raw gradient is never modified. Coupling is applied only to the increments after the update rule calculates them.

For gradient descent,

$$G_x^{(i)}=\frac{\partial f}{\partial x},\qquad G_y^{(i)}=\frac{\partial f}{\partial y},$$

$$\Delta x_i=-\eta G_x^{(i)},\qquad \Delta y_i=-\eta G_y^{(i)}.$$

The applied increments are

$$s_i=0:\quad (\widetilde{\Delta x_i},\widetilde{\Delta y_i})=(\Delta x_i,\Delta y_i),$$

$$s_i=1:\quad (\widetilde{\Delta x_i},\widetilde{\Delta y_i})=(\Delta y_i,\Delta x_i),$$

followed by

$$x_{i+1}=x_i+\widetilde{\Delta x_i},\qquad y_{i+1}=y_i+\widetilde{\Delta y_i}.$$

There is no operation $(x_i,y_i)\leftarrow(y_i,x_i)$.

In [ ]:
import numpy as np
from IPython.display import Markdown, display

from tuple_gradient import (
    build_swap_schedule,
    coupling_coefficient,
    coupled_gradient_descent_step,
    empirical_swap_frequency,
    gradient_descent_increments,
    numerical_gradient,
    run_coupled_gradient_descent,
    swap_increments,
)

np.set_printoptions(precision=4, suppress=True)

## Example 1: Computing $\nu_N$ from a swap-indicator array

For $N=10$, let

$$\mathbf{s}=[0,1,0,0,1,0,1,0,0,1].$$

The empirical coupling coefficient is $C=\nu_N=N^{-1}\sum_i s_i$.

In [ ]:
s_example_1 = np.array([0, 1, 0, 0, 1, 0, 1, 0, 0, 1], dtype=np.int8)
nu_10 = coupling_coefficient(s_example_1)

print('swap indicators:', s_example_1)
print('number of swaps:', int(s_example_1.sum()))
print('nu_10 = C:', nu_10)
assert nu_10 == 0.4

## Example 2: Strong, weak, and periodic coupling

Run A has eight swaps in ten steps; run B has one. A periodic schedule that swaps every third step approaches $C=1/3$ and equals $1/3$ whenever the measured length is a multiple of three.

In [ ]:
s_strong = np.array([1, 1, 0, 1, 1, 1, 0, 1, 1, 1], dtype=np.int8)
s_weak = np.array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0], dtype=np.int8)
s_periodic = np.array(
    [int(iteration % 3 == 0) for iteration in range(1, 31)],
    dtype=np.int8,
)

C_strong = coupling_coefficient(s_strong)
C_weak = coupling_coefficient(s_weak)
C_periodic = coupling_coefficient(s_periodic)

print('strong coupling C:', C_strong)
print('weak coupling C:', C_weak)
print('periodic coupling over 30 steps C:', C_periodic)

assert C_strong == 0.8
assert C_weak == 0.1
assert C_periodic == 1.0 / 3.0

### Optional schedule construction

An explicit indicator array is the primary experimental record. For convenience, `build_swap_schedule(N, C)` creates an evenly distributed schedule when $NC$ is an integer. Its measured coefficient is checked against $C$.

In [ ]:
generated_schedule = np.array(build_swap_schedule(10, 0.4), dtype=np.int8)
print('generated schedule:', generated_schedule)
print('measured C:', coupling_coefficient(generated_schedule))
assert generated_schedule.sum() == 4
assert coupling_coefficient(generated_schedule) == 0.4

## Example 3: Full trajectory for $f(x,y)=x^2y$

The raw partial derivatives are

$$G_x=2xy,\qquad G_y=x^2.$$

We use $\eta=0.1$, initial state $(1,2)$, and explicit indicators $[0,1,0,1,0]$. The implementation differentiates numerically, calculates the raw deltas, swaps only those deltas at steps 2 and 4, and adds them to the current state.

In [ ]:
def objective(x, y):
    return x**2 * y

s_trajectory = np.array([0, 1, 0, 1, 0], dtype=np.int8)
trajectory = run_coupled_gradient_descent(
    objective,
    initial_parameters=(1.0, 2.0),
    learning_rate=0.1,
    swap_indicators=s_trajectory,
    unpack=True,
)

print('x =', np.round(trajectory.x, 4))
print('y =', np.round(trajectory.y, 4))
print('s =', trajectory.swap_indicators)
print('C =', trajectory.coefficient)

In [ ]:
table_lines = [
    '| i | x_i | y_i | G_x | G_y | s_i | delta used for x | delta used for y |',
    '|---:|---:|---:|---:|---:|:---:|---:|---:|',
]

for index, indicator in enumerate(trajectory.swap_indicators):
    x_i, y_i = trajectory.states[index]
    G_x, G_y = trajectory.gradients[index]
    applied_x, applied_y = trajectory.applied_increments[index]
    table_lines.append(
        f'| {index + 1} | {x_i:.4f} | {y_i:.4f} | {G_x:.4f} | {G_y:.4f} | '
        f'{int(indicator)} | {applied_x:.4f} | {applied_y:.4f} |'
    )

final_x, final_y = trajectory.states[-1]
table_lines.append(f'| 6 | {final_x:.4f} | {final_y:.4f} | — | — | — | — | — |')
display(Markdown('\n'.join(table_lines)))

### Verify that only the deltas swap

At step 2 the state is $(0.6,1.9)$. Its ordinary deltas are $(-0.228,-0.036)$. Because $s_2=1$, only those deltas become $(-0.036,-0.228)$, producing $(0.564,1.672)$. The state never becomes $(1.9,0.6)$.

In [ ]:
step_2_index = 1
print('state before step 2: ', trajectory.states[step_2_index])
print('raw step-2 deltas:   ', trajectory.raw_increments[step_2_index])
print('applied step-2 deltas:', trajectory.applied_increments[step_2_index])
print('state after step 2:  ', trajectory.states[step_2_index + 1])

assert np.allclose(trajectory.states[step_2_index], (0.6, 1.9))
assert np.allclose(trajectory.raw_increments[step_2_index], (-0.228, -0.036))
assert np.allclose(trajectory.applied_increments[step_2_index], (-0.036, -0.228))
assert np.allclose(trajectory.states[step_2_index + 1], (0.564, 1.672))
assert not np.allclose(trajectory.states[step_2_index + 1], (1.9, 0.6))

## Example 4: Computing $\nu_N$ directly with NumPy

For a validated binary array, its mean and `count_nonzero / size` both implement $N^{-1}\sum_i s_i$. The library helper additionally rejects empty or non-binary arrays.

In [ ]:
s_numpy = np.array([0, 1, 0, 0, 1, 0, 1, 0, 0, 1], dtype=np.int8)
nu_by_mean = float(np.mean(s_numpy))
nu_by_count = float(np.count_nonzero(s_numpy) / s_numpy.size)
nu_validated = empirical_swap_frequency(s_numpy)

print('np.mean:', nu_by_mean)
print('np.count_nonzero / N:', nu_by_count)
print('validated helper:', nu_validated)
assert nu_by_mean == nu_by_count == nu_validated == 0.4

## Example 5: How gradient trajectories change indirectly under increment coupling

We repeat Example 3 for $N=40$ using four periodic schedules. The raw definitions $G_x=2xy$ and $G_y=x^2$ never change, and the gradient tuple is never swapped. What changes is which **already-calculated delta** is added to each coordinate. That produces a different state trajectory, so gradients evaluated at later states follow different curves.

| Schedule | Indicator pattern | $C=\nu_{40}$ |
|---|---|:---:|
| no swaps | $[0,0,0,0,\ldots]$ | $0$ |
| every 4th step | $[0,0,0,1,0,0,0,1,\ldots]$ | $0.25$ |
| every 2nd step | $[0,1,0,1,0,1,\ldots]$ | $0.5$ |
| every step | $[1,1,1,1,\ldots]$ | $1$ |

In [ ]:
N_EXAMPLE_5 = 40
ETA_EXAMPLE_5 = 0.1
example_5_panels = [
    (0, 'no swaps'),
    (4, 'swap every 4th step'),
    (2, 'swap every 2nd step'),
    (1, 'swap every step'),
]

example_5_runs = []
for period, label in example_5_panels:
    indicators = np.array(
        [
            int(period > 0 and iteration % period == 0)
            for iteration in range(1, N_EXAMPLE_5 + 1)
        ],
        dtype=np.int8,
    )
    result = run_coupled_gradient_descent(
        objective,
        initial_parameters=(1.0, 2.0),
        learning_rate=ETA_EXAMPLE_5,
        swap_indicators=indicators,
        unpack=True,
    )
    example_5_runs.append((period, label, result))

endpoint_lines = [
    '| schedule | C | final x | final y |',
    '|---|---:|---:|---:|',
]
for _, label, result in example_5_runs:
    final_x, final_y = result.states[-1]
    endpoint_lines.append(
        f'| {label} | {result.coefficient:g} | {final_x:.6f} | {final_y:.6f} |'
    )
display(Markdown('\n'.join(endpoint_lines)))

In [ ]:
# Audit every step: applied deltas equal raw deltas for s=0 and only their
# reversed order for s=1. No state row is ever reversed.
for _, _, result in example_5_runs:
    for index, indicator in enumerate(result.swap_indicators):
        expected_delta = (
            result.raw_increments[index][::-1]
            if indicator
            else result.raw_increments[index]
        )
        assert np.allclose(result.applied_increments[index], expected_delta)
        assert np.allclose(
            result.states[index + 1],
            result.states[index] + result.applied_increments[index],
        )
print('All 160 Example-5 updates passed the delta-only swap audit.')

In [ ]:
import matplotlib.pyplot as plt

SURFACE = '#fcfcfb'
TEXT_PRIMARY = '#1a1a19'
TEXT_SECONDARY = '#5f5e56'
GRID = '#e8e7e2'
C_GX = '#2a78d6'
C_GY = '#008300'

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4), sharey=True)
fig.patch.set_facecolor(SURFACE)
iterations = np.arange(1, N_EXAMPLE_5 + 1)

for ax, (_, label, result) in zip(axes, example_5_runs):
    G_x = result.gradients[:, 0]
    G_y = result.gradients[:, 1]
    ax.set_facecolor(SURFACE)
    ax.plot(iterations, G_x, color=C_GX, lw=2)
    ax.plot(iterations, G_y, color=C_GY, lw=2)
    ax.set_title(f'$C = {result.coefficient:g}$', fontsize=12, color=TEXT_PRIMARY, pad=10)
    ax.text(0.5, 1.005, label, transform=ax.transAxes, ha='center',
            va='bottom', fontsize=8.5, color=TEXT_SECONDARY)
    ax.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_color(GRID)
    ax.tick_params(colors=TEXT_SECONDARY, labelsize=9)
    ax.set_xlabel('iteration $i$', fontsize=10, color=TEXT_SECONDARY)
    ax.set_xlim(1, N_EXAMPLE_5)

axes[0].set_ylabel('raw gradient value', fontsize=10, color=TEXT_SECONDARY)
axes[0].set_ylim(bottom=0)
fig.legend(
    handles=[
        plt.Line2D([], [], color=C_GX, lw=2, label='$G_x^{(i)}=2x_i y_i$'),
        plt.Line2D([], [], color=C_GY, lw=2, label='$G_y^{(i)}=x_i^2$'),
    ],
    loc='upper right', ncol=2, frameon=False, fontsize=10,
    bbox_to_anchor=(0.99, 1.02), labelcolor=TEXT_PRIMARY,
)
fig.suptitle(
    'Raw gradient trajectories after frequency-coupled increments',
    x=0.01, ha='left', fontsize=13, color=TEXT_PRIMARY,
)
fig.tight_layout(rect=[0, 0, 1, 0.90])
fig.savefig(
    'coupling-frequency-gradients.png',
    dpi=160, facecolor=SURFACE, bbox_inches='tight',
)
plt.show()

In [ ]:
import pandas as pd

gradient_columns = {}
for _, _, result in example_5_runs:
    label = f'C={result.coefficient:g}'
    gradient_columns[f'{label} Gx'] = result.gradients[:, 0]
    gradient_columns[f'{label} Gy'] = result.gradients[:, 1]

gradient_table = pd.DataFrame(
    gradient_columns,
    index=np.arange(1, N_EXAMPLE_5 + 1),
)
gradient_table.index.name = 'iteration'
gradient_table

### Example 5 interpretation

Increasing $C$ redirects the large **increment generated from** $G_x$ away from $x$ and into $y$ more often. This changes the future states at which the unchanged gradient formulas are evaluated. Consequently the plotted gradient trajectories persist longer and the decay burden shifts from $x$ toward $y$. The gradients are not themselves exchanged or redefined.

## Verification suite

These checks separately verify the raw derivative, ordinary deltas, delta-only swap, complete trajectory, and empirical coefficient.

In [ ]:
# Raw differentiation always stays in (G_x, G_y) order.
raw_gradient = numerical_gradient(objective, (1.0, 2.0), unpack=True)
assert np.allclose(raw_gradient, (4.0, 1.0), atol=1e-8)

# The ordinary deltas are calculated before coupling.
raw_delta = gradient_descent_increments(raw_gradient, 0.1)
assert np.allclose(raw_delta, (-0.4, -0.1))

# s=0 preserves deltas; s=1 reverses only deltas.
assert np.allclose(swap_increments(raw_delta, 0), (-0.4, -0.1))
assert np.allclose(swap_increments(raw_delta, 1), (-0.1, -0.4))

expected_states = np.array([
    [1.0000, 2.0000],
    [0.6000, 1.9000],
    [0.5640, 1.6720],
    [0.3754, 1.6402],
    [0.3613, 1.5170],
    [0.2517, 1.5040],
])
assert np.allclose(trajectory.states, expected_states, atol=5e-5)
assert trajectory.coefficient == 2 / 5

# Example 5 endpoint and coefficient checks.
expected_example_5_coefficients = np.array([0.0, 0.25, 0.5, 1.0])
expected_example_5_endpoints = np.array([
    [0.000000, 1.841126],
    [0.000002, 1.724836],
    [0.001498, 1.243038],
    [0.193474, 0.068278],
])
actual_coefficients = np.array([run.coefficient for _, _, run in example_5_runs])
actual_endpoints = np.array([run.states[-1] for _, _, run in example_5_runs])
assert np.allclose(actual_coefficients, expected_example_5_coefficients)
assert np.allclose(actual_endpoints, expected_example_5_endpoints, atol=1e-6)

print('All Examples 1–5 passed: only increments are swapped.')

## Result

The complete Examples 1–5 suite keeps three objects distinct: the state $(x_i,y_i)$, the raw gradient $(G_x,G_y)$, and the update increments $(\Delta x_i,\Delta y_i)$. The indicator acts only on the third object. It reproduces the supplied five-step trajectory, NumPy frequency calculation, all four Example 5 coupling regimes, their endpoints, gradient table, and comparison plot.